# Frozen hydropower ensemble

**Group number:** TODO  
**Full names and student IDs:** TODO  
**Kaggle team name:** TODO

This development notebook inlines every selected model pipeline. It requires only
the supplied raw data and installed packages, and trains sequentially on four CPU
threads. No source helper, other notebook, saved model or previous prediction CSV
is a runtime dependency. Temporary component artifacts are generated during execution.

Weights and configurations were frozen using purged 2020/2021 development folds.
They are exploratory selections, not independent confirmation or Kaggle scores.
`check` audits inputs without fitting; `submit` refits every nonzero-weight model on
all labeled cases and exports the blended probabilities. Final course delivery still
requires student metadata, pinned environment, the companion notebook and report.


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import hashlib
import numpy as np
import pandas as pd

ENSEMBLE_MODE = "check"  # check or submit; submit refits all selected components
ENSEMBLE_WEIGHTS = {'catboost': 0.5625, 'lightgbm': 0.1875, 'tcn': 0.25}
if ENSEMBLE_MODE not in {"check", "submit"}:
    raise ValueError("Ensemble mode must be check or submit.")
ENSEMBLE_STARTED = perf_counter()
COMPONENT_METADATA = {}
COMPONENT_FRAMES = {}


## Catboost component

Frozen experiment: `decision_rsm_logloss`. The following pipeline is inlined.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import platform
import re

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd()
RUN_MODE = ENSEMBLE_MODE
SEED = 42
THREADS = 4
HOURS = 168
# Compare one change at a time on the same development folds.
EXPERIMENT = 'decision_rsm_logloss'
EXPERIMENTS = {
    "decision_rsm_logloss": {"features": "hydro-decision-v1", "stopping_metric": "Logloss",
                             "decision_groups": ["economics", "water_balance"],
                             "parameter_overrides": {"rsm": 0.8}},
    "decision_logloss": {"features": "hydro-decision-v1", "stopping_metric": "Logloss",
                         "decision_groups": ["economics", "water_balance"]},
    "economics_logloss": {"features": "hydro-economics-v1", "stopping_metric": "Logloss",
                          "decision_groups": ["economics"]},
    "water_balance_logloss": {"features": "hydro-water-balance-v1", "stopping_metric": "Logloss",
                              "decision_groups": ["water_balance"]},
    "baseline_auc": {"features": "hydro-context-v1", "stopping_metric": "AUC"},
    "probability_logloss": {"features": "hydro-context-v1", "stopping_metric": "Logloss"},
    "shared_logloss": {"features": "hydro-context-v1", "stopping_metric": "Logloss", "strategy": "shared"},
    "upstream_logloss": {"features": "hydro-context-v2", "stopping_metric": "Logloss"},
}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError("Unknown experiment preset.")
FEATURE_VERSION = EXPERIMENTS[EXPERIMENT]["features"]
DECISION_GROUPS = set(EXPERIMENTS[EXPERIMENT].get("decision_groups", []))
MODEL_STRATEGY = EXPERIMENTS[EXPERIMENT].get("strategy", "per_generator")
# This limits one raw numeric allocation, not total process memory.
SHARED_MAX_MATRIX_GIB = 6.0
FOLDS = [("2020-01-01", "2021-01-01"), ("2021-01-01", "2022-01-01")]
HOLDOUT_START = pd.Timestamp("2022-01-01", tz="UTC")
EARLY_STOPPING = 100
MAX_ITERATIONS = 1500
BASE_PARAMETERS = {
    "loss_function": "Logloss",
    "eval_metric": EXPERIMENTS[EXPERIMENT]["stopping_metric"],
    "custom_metric": ["AUC"] if EXPERIMENTS[EXPERIMENT]["stopping_metric"] == "Logloss" else ["Logloss"],
    "depth": 7,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
    "border_count": 128,
    "task_type": "CPU",
}
# Apply only the parameter changes declared by the selected experiment.
BASE_PARAMETERS.update(EXPERIMENTS[EXPERIMENT].get("parameter_overrides", {}))
if MODEL_STRATEGY == "shared":
    # Treat generator identity as a category; avoid target-statistic encodings here.
    BASE_PARAMETERS["one_hot_max_size"] = 20

# Paste the complete dictionary printed by a completed validate run here.
# Keep the previously validated configuration for reference; do not overwrite its results.
REFERENCE_CONFIG = {'feature_version': 'hydro-context-v1', 'feature_signature': '4d09eef00a583fad18e6d8cd0f361f2d06a4e9fcfecc5e904dba743f664df16e', 'seed': 42, 'parameters': {'loss_function': 'Logloss', 'eval_metric': 'AUC', 'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 5.0, 'random_strength': 0.5, 'bootstrap_type': 'Bernoulli', 'subsample': 0.8, 'border_count': 128, 'task_type': 'CPU'}, 'iterations': {'Hogga_G1': 217, 'Tokke_G1': 91, 'Tokke_G2': 484, 'Tokke_G3': 229, 'Tokke_G4': 123, 'Lio_G1': 147, 'Byrte_G1': 173, 'Vinje_G1': 373, 'Vinje_G2': 332, 'Vinje_G3': 231, 'Haukeli_G1': 23, 'Songa_G1': 486, 'Kjela_G1': 212, 'Vesle Kjela_G1': 133}}
# Frozen configuration from the completed probability_logloss validation run.
PROBABILITY_REFERENCE_CONFIG = {'experiment': 'probability_logloss',
 'feature_version': 'hydro-context-v1',
 'feature_signature': '4d09eef00a583fad18e6d8cd0f361f2d06a4e9fcfecc5e904dba743f664df16e',
 'seed': 42,
 'parameters': {'loss_function': 'Logloss',
                'eval_metric': 'Logloss',
                'custom_metric': ['AUC'],
                'depth': 7,
                'learning_rate': 0.05,
                'l2_leaf_reg': 5.0,
                'random_strength': 0.5,
                'bootstrap_type': 'Bernoulli',
                'subsample': 0.8,
                'border_count': 128,
                'task_type': 'CPU'},
 'iterations': {'Hogga_G1': 223,
                'Tokke_G1': 147,
                'Tokke_G2': 228,
                'Tokke_G3': 147,
                'Tokke_G4': 188,
                'Lio_G1': 149,
                'Byrte_G1': 438,
                'Vinje_G1': 258,
                'Vinje_G2': 175,
                'Vinje_G3': 132,
                'Haukeli_G1': 21,
                'Songa_G1': 433,
                'Kjela_G1': 162,
                'Vesle Kjela_G1': 240}}

# Frozen decision-feature reference that produced the reported Kaggle score 0.98679.
DECISION_REFERENCE_CONFIG = {'experiment': 'decision_logloss',
 'model_strategy': 'per_generator',
 'feature_version': 'hydro-decision-v1',
 'feature_signature': '413b34b792c90dc7fb586250679bd5438ef0dc4e90ef8b07a8bdb6331b6c2af9',
 'seed': 42,
 'parameters': {'loss_function': 'Logloss',
                'eval_metric': 'Logloss',
                'custom_metric': ['AUC'],
                'depth': 7,
                'learning_rate': 0.05,
                'l2_leaf_reg': 5.0,
                'random_strength': 0.5,
                'bootstrap_type': 'Bernoulli',
                'subsample': 0.8,
                'border_count': 128,
                'task_type': 'CPU'},
 'iterations': {'Hogga_G1': 191,
                'Tokke_G1': 212,
                'Tokke_G2': 319,
                'Tokke_G3': 260,
                'Tokke_G4': 186,
                'Lio_G1': 65,
                'Byrte_G1': 547,
                'Vinje_G1': 245,
                'Vinje_G2': 181,
                'Vinje_G3': 332,
                'Haukeli_G1': 44,
                'Songa_G1': 168,
                'Kjela_G1': 452,
                'Vesle Kjela_G1': 750}}

# Frozen RSM configuration from the completed 2020/2021 validation run.
SELECTED_CONFIG = {'experiment': 'decision_rsm_logloss',
 'model_strategy': 'per_generator',
 'feature_version': 'hydro-decision-v1',
 'feature_signature': '413b34b792c90dc7fb586250679bd5438ef0dc4e90ef8b07a8bdb6331b6c2af9',
 'seed': 42,
 'parameters': {'loss_function': 'Logloss',
                'eval_metric': 'Logloss',
                'custom_metric': ['AUC'],
                'depth': 7,
                'learning_rate': 0.05,
                'l2_leaf_reg': 5.0,
                'random_strength': 0.5,
                'bootstrap_type': 'Bernoulli',
                'subsample': 0.8,
                'border_count': 128,
                'task_type': 'CPU',
                'rsm': 0.8},
 'iterations': {'Hogga_G1': 353,
                'Tokke_G1': 192,
                'Tokke_G2': 314,
                'Tokke_G3': 151,
                'Tokke_G4': 156,
                'Lio_G1': 125,
                'Byrte_G1': 294,
                'Vinje_G1': 226,
                'Vinje_G2': 341,
                'Vinje_G3': 364,
                'Haukeli_G1': 42,
                'Songa_G1': 190,
                'Kjela_G1': 223,
                'Vesle Kjela_G1': 439}}

if RUN_MODE not in {"check", "smoke", "validate", "holdout", "submit"}:
    raise ValueError("Unknown RUN_MODE.")
if not (ROOT / "prediction_mapping.csv").is_file():
    raise FileNotFoundError("Run the notebook from the project root with the supplied data present.")
if not 1 <= THREADS <= 4:
    raise ValueError("Use between 1 and 4 CPU threads for the course reproduction budget.")
NOTEBOOK_STARTED = perf_counter()
VERSIONS = {"python": platform.python_version()}
for package in ("numpy", "pandas", "PyYAML", "catboost", "scikit-learn", "matplotlib"):
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = "not installed"
print("Mode:", RUN_MODE, "Experiment:", EXPERIMENT, "Features:", FEATURE_VERSION)
print("Package versions:", VERSIONS)
if RUN_MODE != "check":
    missing = [p for p in ("catboost", "scikit-learn") if VERSIONS[p] == "not installed"]
    if missing:
        raise ImportError(f"Install requirements.txt into this kernel's environment first: {missing}")

In [ ]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")
display(pd.DataFrame({"generator": GENERATORS,
    "training_on_fraction": [float(labels_for(train_cases, g).mean()) for g in GENERATORS]}))

In [ ]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")
display(pd.DataFrame({"generator": GENERATORS,
                      "intake_reservoirs": [", ".join(FEEDERS[g]) for g in GENERATORS]}))
display(pd.DataFrame([{"input": k, "rows_in_task_range": len(v), "columns": len(v.columns),
                       "start": str(v.index.min()), "end": str(v.index.max())}
                      for k, v in tables.items()]))

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))


In [ ]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32, copy=False)

In [ ]:
def chronological_split(start, end):
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    if start >= end or end > HOLDOUT_START:
        raise ValueError("Development folds must end by the reserved holdout start.")
    horizon_end = train_cases["starttime"] + pd.Timedelta(hours=HOURS)
    fitting = train_cases.loc[horizon_end <= start].copy()
    validation = train_cases.loc[(train_cases["starttime"] >= start) & (horizon_end <= end)].copy()
    if fitting.empty or validation.empty:
        raise ValueError("Empty chronological fold.")
    if (fitting["starttime"] + pd.Timedelta(hours=HOURS)).max() > validation["starttime"].min():
        raise ValueError("Training and validation horizons overlap.")
    return fitting, validation

def even_sample(cases, maximum):
    indices = np.linspace(0, len(cases) - 1, min(maximum, len(cases)), dtype=int)
    return cases.iloc[indices].copy()

def assemble_predictions(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("Predictions must cover all 14 generators.")
    values = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for generator in GENERATORS:
        probability = np.asarray(predictions[generator])
        if probability.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {generator}")
        values[:, TARGET_POSITIONS[generator]] = probability
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    result = pd.DataFrame(values, columns=LABELS)
    result.insert(0, "Run No", cases["Run No"].to_numpy())
    return result

display(pd.DataFrame([{"validation_start": start, "validation_end_exclusive": end,
                       "training_cases": len(chronological_split(start, end)[0]),
                       "validation_cases": len(chronological_split(start, end)[1])}
                      for start, end in FOLDS]))
probe_cases = pd.concat([train_cases.iloc[[0, 1]], test_cases.iloc[[0, -1]]], ignore_index=True)
probe = feature_matrix(probe_cases)
probe_single = feature_matrix(probe_cases.iloc[[1]])
np.testing.assert_allclose(probe.iloc[HOURS:2 * HOURS].to_numpy(), probe_single.to_numpy())
for i, start in enumerate(probe_cases["starttime"]):
    rows = slice(i * HOURS, (i + 1) * HOURS)
    times = pd.date_range(start, periods=HOURS, freq="h")
    np.testing.assert_allclose(probe.iloc[rows]["price"], lookup("prices", times).reshape(-1))
    for reservoir in RESERVOIRS:
        np.testing.assert_allclose(probe.iloc[rows][f"initial_volume__{reservoir}"], tables["volumes"].at[start, reservoir])
        np.testing.assert_allclose(probe.iloc[rows][f"terminal_value__{reservoir}"], tables["water_values"].at[start + pd.Timedelta(hours=HOURS), reservoir])
for g in GENERATORS:
    generator_features(probe, g)
    truth = labels_for(train_cases.iloc[:2], g).reshape(2, HOURS)
    np.testing.assert_array_equal(truth[:, [0, 24, 167]], train_cases.iloc[:2][[TARGET_COLUMNS[g][h] for h in (0, 24, 167)]])
roundtrip = assemble_predictions(train_cases.iloc[:2],
    {g: labels_for(train_cases.iloc[:2], g).reshape(2, HOURS) for g in GENERATORS})
np.testing.assert_array_equal(roundtrip[LABELS], train_cases.iloc[:2][LABELS])
NUMERIC_FEATURE_NAMES = generator_features(probe, GENERATORS[0]).columns.tolist()
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + (["generator_id"] if MODEL_STRATEGY == "shared" else [])
MODEL_KEYS = ["shared"] if MODEL_STRATEGY == "shared" else GENERATORS
FEATURE_SIGNATURE = hashlib.sha256(json.dumps(FEATURE_NAMES).encode()).hexdigest()
if any(c in FEATURE_NAMES for c in ("Run No", "starttime", "total_calculation_time")):
    raise ValueError("Metadata leaked into predictors.")
bytes_per_case = HOURS * len(NUMERIC_FEATURE_NAMES) * np.dtype("float32").itemsize
print(f"Data checks passed: {len(FEATURE_NAMES)} features per generator-hour.")
print(f"One full training feature matrix: approximately {len(train_cases) * bytes_per_case / 2**20:.0f} MiB.")
print("This is not peak memory: copies, CatBoost pools, and model state add overhead.")
del probe, probe_single, roundtrip
if MODEL_STRATEGY == "shared":
    shared_rows = len(train_cases) * HOURS * len(GENERATORS)
    raw_gib = shared_rows * len(NUMERIC_FEATURE_NAMES) * 4 / 2**30
    print(f"Shared full-data fit: {shared_rows:,} rows, {len(NUMERIC_FEATURE_NAMES)} numeric features + generator_id.")
    print(f"Raw numeric allocation: {raw_gib:.2f} GiB; CatBoost and preprocessing require additional RAM.")
    print("Run smoke before full validation. Shared-model runtime has not been measured.")


In [ ]:
def safe_auc(actual, predicted):
    from sklearn.metrics import roc_auc_score
    actual = np.asarray(actual).reshape(-1)
    if np.unique(actual).size < 2:
        return None
    return float(roc_auc_score(actual, np.asarray(predicted).reshape(-1)))

def evaluate(cases, predictions):
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    frame = assemble_predictions(cases, predictions)
    actual = cases[LABELS].to_numpy(dtype=np.uint8)
    predicted = frame[LABELS].to_numpy()
    overall = float(roc_auc_score(actual, predicted, average="micro"))
    detail = pd.DataFrame([{"generator": g,
        "auc": safe_auc(labels_for(cases, g), predictions[g]),
        "logloss": float(log_loss(labels_for(cases, g), predictions[g].reshape(-1), labels=[0, 1])),
        "brier": float(brier_score_loss(labels_for(cases, g), predictions[g].reshape(-1))),
        "observed_on_fraction": float(labels_for(cases, g).mean()),
        "mean_prediction": float(predictions[g].mean())} for g in GENERATORS])
    return overall, detail, frame

def seasonal_predictions(fitting, validation):
    groups = fitting[LABELS].groupby(fitting["starttime"].dt.month).mean()
    if not set(validation["starttime"].dt.month).issubset(groups.index):
        raise ValueError("The seasonal reference lacks a validation month in training.")
    values = groups.loc[validation["starttime"].dt.month]
    return {g: values[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32) for g in GENERATORS}

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def fit_and_predict(fitting, evaluation, parameters, iterations, folder, *, early_stop):
    if MODEL_STRATEGY == "shared":
        return fit_shared_and_predict(fitting, evaluation, parameters, iterations, folder, early_stop=early_stop)
    from catboost import CatBoostClassifier
    started = perf_counter()
    folder.mkdir(parents=True, exist_ok=False)
    X_fit_base, X_eval_base = feature_matrix(fitting), feature_matrix(evaluation)
    feature_seconds = perf_counter() - started
    predicted, records, importance, curves = {}, [], [], {}
    for generator in GENERATORS:
        generator_started = perf_counter()
        X_fit = generator_features(X_fit_base, generator)
        X_eval = generator_features(X_eval_base, generator)
        if X_fit.columns.tolist() != FEATURE_NAMES or X_eval.columns.tolist() != FEATURE_NAMES:
            raise ValueError("Feature schema changed between data partitions.")
        y = labels_for(fitting, generator)
        limit = int(iterations[generator] if isinstance(iterations, dict) else iterations)
        if limit < 1:
            raise ValueError("Tree counts must be positive.")
        model = None
        if np.unique(y).size == 1:
            probability = np.full(len(evaluation) * HOURS, float(y[0]), dtype=np.float32)
            tree_count = 1
            constant = float(y[0])
            write_json(folder / f"{generator}_constant.json", {"probability": constant})
        else:
            constant = None
            model = CatBoostClassifier(**parameters, iterations=limit, random_seed=SEED,
                                      thread_count=THREADS, allow_writing_files=False)
            options = {"verbose": 100}
            if early_stop:
                y_eval = labels_for(evaluation, generator)
                # Logloss can evaluate one-class subsets; AUC cannot.
                if parameters.get("eval_metric") != "AUC" or np.unique(y_eval).size == 2:
                    options.update(eval_set=(X_eval, y_eval),
                                   early_stopping_rounds=EARLY_STOPPING, use_best_model=True)
            print(f"Training {generator}: {len(y):,} hourly rows, at most {limit} trees", flush=True)
            model.fit(X_fit, y, **options)
            if list(model.classes_) != [0, 1]:
                raise ValueError("Unexpected CatBoost class order.")
            probability = model.predict_proba(X_eval, thread_count=THREADS)[:, 1].astype(np.float32)
            tree_count = int(model.tree_count_)
            model.save_model(str(folder / f"{generator}.cbm"))
            curves[generator] = model.get_evals_result()
            importance.extend({"generator": generator, "feature": name, "importance": float(value)}
                              for name, value in zip(FEATURE_NAMES, model.get_feature_importance(thread_count=THREADS)))
        predicted[generator] = probability.reshape(len(evaluation), HOURS)
        records.append({"generator": generator, "trees": tree_count,
                        "constant_probability": constant,
                        "seconds": perf_counter() - generator_started})
        pd.DataFrame(records).to_csv(folder / "training_progress.csv", index=False)
        del X_fit, X_eval, model
        gc.collect()
    pd.DataFrame(importance, columns=["generator", "feature", "importance"]).to_csv(folder / "feature_importance.csv", index=False)
    write_json(folder / "learning_curves.json", curves)
    write_json(folder / "timing.json", {"feature_seconds": feature_seconds,
                                       "total_seconds": perf_counter() - started})
    del X_fit_base, X_eval_base
    gc.collect()
    return predicted, records, importance

In [ ]:
def shared_arrays(cases, *, include_labels):
    """Generator-major rows, then input case order, then horizon hour 0..167."""
    rows_per_generator = len(cases) * HOURS
    total_rows = rows_per_generator * len(GENERATORS)
    required_gib = total_rows * len(NUMERIC_FEATURE_NAMES) * 4 / 2**30
    if required_gib > SHARED_MAX_MATRIX_GIB:
        raise MemoryError(f"Shared numeric matrix needs {required_gib:.2f} GiB, above SHARED_MAX_MATRIX_GIB. No rows were sampled.")
    base = feature_matrix(cases)
    numeric = np.empty((total_rows, len(NUMERIC_FEATURE_NAMES)), dtype=np.float32)
    categorical = np.empty((total_rows, 1), dtype=object)
    labels = np.empty(total_rows, dtype=np.uint8) if include_labels else None
    for i, generator in enumerate(GENERATORS):
        section = slice(i * rows_per_generator, (i + 1) * rows_per_generator)
        block = generator_features(base, generator)
        if block.columns.tolist() != NUMERIC_FEATURE_NAMES:
            raise ValueError("Shared numeric feature schema mismatch.")
        numeric[section] = block.to_numpy(dtype=np.float32, copy=False)
        categorical[section, 0] = generator.encode("utf-8")
        if include_labels:
            labels[section] = labels_for(cases, generator)
        del block
    del base
    return numeric, categorical, labels

def split_shared_predictions(values, cases):
    values = np.asarray(values, dtype=np.float32)
    expected = len(GENERATORS) * len(cases) * HOURS
    if values.shape != (expected,):
        raise ValueError(f"Expected {expected} shared probabilities, got {values.shape}.")
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Shared predictions must be finite probabilities in [0, 1].")
    cube = values.reshape(len(GENERATORS), len(cases), HOURS)
    return {g: cube[i] for i, g in enumerate(GENERATORS)}

def shared_pool(cases, parameters, *, include_labels, input_borders=None):
    from catboost import FeaturesData, Pool
    numeric, categorical, labels = shared_arrays(cases, include_labels=include_labels)
    data = FeaturesData(num_feature_data=numeric, cat_feature_data=categorical,
                        num_feature_names=NUMERIC_FEATURE_NAMES,
                        cat_feature_names=["generator_id"])
    pool = Pool(data, label=labels, thread_count=THREADS)
    if pool.get_feature_names() != FEATURE_NAMES or pool.get_cat_feature_indices() != [len(NUMERIC_FEATURE_NAMES)]:
        raise ValueError("Incorrect shared Pool feature names or categorical column.")
    # Fit numeric bin boundaries on training inputs only. Evaluation reuses them.
    # Quantize and release the raw arrays before constructing the next partition.
    if input_borders is None:
        pool.quantize(border_count=parameters.get("border_count", 128), random_seed=SEED, task_type="CPU")
    else:
        pool.quantize(input_borders=str(input_borders), task_type="CPU")
    del data, numeric, categorical, labels
    gc.collect()
    return pool

def fit_shared_and_predict(fitting, evaluation, parameters, iterations, folder, *, early_stop):
    from catboost import CatBoostClassifier
    started = perf_counter()
    folder.mkdir(parents=True, exist_ok=False)
    limit = int(iterations["shared"] if isinstance(iterations, dict) else iterations)
    if limit < 1:
        raise ValueError("Shared tree count must be positive.")
    print(f"Preparing one shared model: {len(fitting) * HOURS * len(GENERATORS):,} training rows", flush=True)
    fitting_pool = shared_pool(fitting, parameters, include_labels=True)
    borders = folder / "numeric_borders.tsv"
    fitting_pool.save_quantization_borders(str(borders))
    # In holdout/submit, evaluation labels are not read during fitting or prediction.
    evaluation_pool = shared_pool(evaluation, parameters, include_labels=early_stop, input_borders=borders)
    feature_seconds = perf_counter() - started
    model = CatBoostClassifier(**parameters, iterations=limit, random_seed=SEED,
                              thread_count=THREADS, allow_writing_files=False)
    options = {"verbose": 100}
    if early_stop:
        options.update(eval_set=evaluation_pool, early_stopping_rounds=EARLY_STOPPING, use_best_model=True)
    print(f"Training shared CatBoost: at most {limit} trees", flush=True)
    model.fit(fitting_pool, **options)
    if list(model.classes_) != [0, 1]:
        raise ValueError("Unexpected shared-model class order.")
    probabilities = model.predict_proba(evaluation_pool, thread_count=THREADS)[:, 1]
    predictions = split_shared_predictions(probabilities, evaluation)
    model.save_model(str(folder / "shared.cbm"))
    record = {"generator": "shared", "trees": int(model.tree_count_),
              "constant_probability": None, "seconds": perf_counter() - started}
    importance = [{"generator": "shared", "feature": name, "importance": float(value)}
                  for name, value in zip(FEATURE_NAMES, model.get_feature_importance(thread_count=THREADS))]
    pd.DataFrame([record]).to_csv(folder / "training_progress.csv", index=False)
    pd.DataFrame(importance).to_csv(folder / "feature_importance.csv", index=False)
    write_json(folder / "learning_curves.json", {"shared": model.get_evals_result()})
    write_json(folder / "timing.json", {"feature_seconds": feature_seconds,
                                       "total_seconds": perf_counter() - started})
    del fitting_pool, evaluation_pool, model
    gc.collect()
    return predictions, [record], importance

if MODEL_STRATEGY == "shared":
    # Small non-training checks: preserve case order and every generator/hour label.
    sample = train_cases.iloc[[1, 0]].copy()
    numeric, category, labels = shared_arrays(sample, include_labels=True)
    expected_labels = np.concatenate([labels_for(sample, g) for g in GENERATORS])
    np.testing.assert_array_equal(labels, expected_labels)
    for i, g in enumerate(GENERATORS):
        section = slice(i * len(sample) * HOURS, (i + 1) * len(sample) * HOURS)
        assert (category[section, 0] == g.encode("utf-8")).all()
    roundtrip = assemble_predictions(sample, split_shared_predictions(labels.astype(np.float32), sample))
    np.testing.assert_array_equal(roundtrip[LABELS], sample[LABELS])
    _, _, no_labels = shared_arrays(test_cases.iloc[:1], include_labels=False)
    assert no_labels is None
    print("Shared row order, categorical generator IDs, and target roundtrip checks passed.")
    del sample, numeric, category, labels, expected_labels, roundtrip, no_labels


In [ ]:
RUN_DIRECTORY = None
run_summary = None
importance_rows = []
if RUN_MODE == "check":
    print("Check mode complete. No models trained and no submission written.")
else:
    if RUN_MODE in {"holdout", "submit"}:
        if not isinstance(SELECTED_CONFIG, dict):
            raise ValueError("Paste a completed validate run's configuration into SELECTED_CONFIG first.")
        if SELECTED_CONFIG.get("feature_version") != FEATURE_VERSION or SELECTED_CONFIG.get("feature_signature") != FEATURE_SIGNATURE:
            raise ValueError("Frozen configuration and current features do not match.")
        if SELECTED_CONFIG.get("experiment", "baseline_auc") != EXPERIMENT:
            raise ValueError("Set EXPERIMENT to the frozen configuration's experiment name before holdout or submit.")
        if SELECTED_CONFIG.get("seed") != SEED:
            raise ValueError("Use the same random seed as the frozen configuration.")
        if SELECTED_CONFIG.get("model_strategy", "per_generator") != MODEL_STRATEGY:
            raise ValueError("Frozen configuration uses a different model strategy.")
        if set(SELECTED_CONFIG.get("iterations", {})) != set(MODEL_KEYS):
            raise ValueError(f"Frozen configuration must contain tree counts for {MODEL_KEYS}.")
        if any(not isinstance(v, int) or v < 1 for v in SELECTED_CONFIG["iterations"].values()):
            raise ValueError("Frozen tree counts must be positive integers.")
        parameters = dict(SELECTED_CONFIG["parameters"])
        if parameters.get("task_type") != "CPU":
            raise ValueError("This notebook's reproducible configuration uses CPU training.")
    else:
        parameters = dict(BASE_PARAMETERS)

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIRECTORY = ROOT / "outputs" / "catboost" / f"{stamp}_{RUN_MODE}_{EXPERIMENT}"
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    run_summary = {"status": "incomplete", "mode": RUN_MODE, "experiment": EXPERIMENT, "versions": VERSIONS,
        "seed": SEED, "threads": THREADS, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION,
        "feature_signature": FEATURE_SIGNATURE, "features": FEATURE_NAMES,
        "parameters": parameters, "folds": FOLDS,
        "maximum_iterations": MAX_ITERATIONS, "early_stopping_rounds": EARLY_STOPPING,
        "selected_configuration": SELECTED_CONFIG,
        "input_sha256": {k: file_hash(v) for k, v in RAW_FILES.items()}}
    write_json(RUN_DIRECTORY / "run.json", run_summary)

    if RUN_MODE in {"smoke", "validate"}:
        fold_results, iteration_records = [], []
        active_folds = FOLDS[-1:] if RUN_MODE == "smoke" else FOLDS
        for start, end in active_folds:
            fitting, validation = chronological_split(start, end)
            if RUN_MODE == "smoke":
                fitting, validation = even_sample(fitting, 96), even_sample(validation, 48)
            folder = RUN_DIRECTORY / f"fold_{start[:4]}"
            predicted, records, importance = fit_and_predict(
                fitting, validation, parameters, 40 if RUN_MODE == "smoke" else MAX_ITERATIONS,
                folder, early_stop=True)
            auc, detail, prediction_frame = evaluate(validation, predicted)
            baseline_auc, _, _ = evaluate(validation, seasonal_predictions(fitting, validation))
            detail.to_csv(folder / "per_generator_metrics.csv", index=False)
            prediction_frame.to_csv(folder / "validation_predictions.csv", index=False)
            iteration_records.extend(records)
            importance_rows.extend(dict(row, fold=start[:4]) for row in importance)
            fold_results.append({"validation_start": start, "validation_end_exclusive": end,
                                 "training_cases": len(fitting), "validation_cases": len(validation),
                                 "micro_auc": auc, "seasonal_micro_auc": baseline_auc,
                                 "mean_generator_logloss": float(detail["logloss"].mean()),
                                 "mean_generator_brier": float(detail["brier"].mean())})
            print(f"{start[:4]} micro ROC-AUC: {auc:.6f}; same-fold seasonal reference: {baseline_auc:.6f}")
            display(detail)
            del predicted, prediction_frame
        run_summary["development_results"] = fold_results
        display(pd.DataFrame(fold_results))
        if RUN_MODE == "validate":
            selected = {"experiment": EXPERIMENT, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION, "feature_signature": FEATURE_SIGNATURE,
                        "seed": SEED, "parameters": parameters,
                        "iterations": {g: max(1, int(np.median([r["trees"] for r in iteration_records if r["generator"] == g]))) for g in MODEL_KEYS}}
            write_json(RUN_DIRECTORY / "selected_configuration.json", selected)
            print("Copy this complete dictionary into SELECTED_CONFIG before holdout or submit:")
            print(repr(selected))
            if any(r["trees"] >= MAX_ITERATIONS for r in iteration_records):
                print("Some models reached MAX_ITERATIONS. Inspect their learning curves before freezing the budget.")
    elif RUN_MODE == "holdout":
        fitting = train_cases.loc[train_cases["starttime"] + pd.Timedelta(hours=HOURS) <= HOLDOUT_START]
        holdout = train_cases.loc[train_cases["starttime"] >= HOLDOUT_START]
        if fitting.empty or holdout.empty:
            raise ValueError("Empty holdout partition.")
        predicted, _, importance_rows = fit_and_predict(fitting, holdout, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "holdout", early_stop=False)
        auc, detail, prediction_frame = evaluate(holdout, predicted)
        prediction_frame.to_csv(RUN_DIRECTORY / "holdout_predictions.csv", index=False)
        detail.to_csv(RUN_DIRECTORY / "holdout_metrics.csv", index=False)
        run_summary["holdout_micro_auc"] = auc
        print(f"Frozen-configuration holdout micro ROC-AUC: {auc:.6f}")
        display(detail)
    elif RUN_MODE == "submit":
        predicted, _, importance_rows = fit_and_predict(train_cases, test_cases, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "full_training", early_stop=False)
        submission = assemble_predictions(test_cases, predicted)
        if not submission.columns.equals(template.columns) or not submission["Run No"].equals(template["Run No"]):
            raise ValueError("Submission IDs or column order differ from the template.")
        if submission.shape != template.shape:
            raise ValueError("Submission shape differs from the template.")
        destination = RUN_DIRECTORY / "catboost_submission.csv"
        submission.to_csv(destination, index=False)
        reloaded = pd.read_csv(destination)
        np.testing.assert_allclose(reloaded[LABELS], submission[LABELS], rtol=1e-6, atol=1e-8)
        np.testing.assert_array_equal(reloaded["Run No"], template["Run No"])
        run_summary["submission"] = str(destination.relative_to(ROOT))
        print(f"Verified submission saved: {destination}")

    run_summary["status"] = "complete"
    run_summary["total_seconds"] = perf_counter() - NOTEBOOK_STARTED
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    (RUN_DIRECTORY / "environment.txt").write_text(
        "\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n", encoding="utf-8")
    print(f"Recorded runtime so far: {run_summary['total_seconds'] / 60:.1f} minutes")
    print("Run artifacts:", RUN_DIRECTORY)

In [ ]:

if ENSEMBLE_MODE == "submit":
    if run_summary["status"] != "complete" or run_summary["mode"] != "submit":
        raise ValueError("Component did not complete a full-data refit.")
    COMPONENT_METADATA['catboost'] = run_summary
    COMPONENT_FRAMES['catboost'] = pd.read_csv(run_summary["submission"])


## Lightgbm component

Frozen experiment: `decision_neutral_logloss`. The following pipeline is inlined.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import platform
import re

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd()
RUN_MODE = ENSEMBLE_MODE
EXPERIMENT = 'decision_neutral_logloss'
MODEL_FAMILY = "lightgbm"
MODEL_STRATEGY = "per_generator"
SEED = 42
THREADS = 4
HOURS = 168
EXPERIMENTS = {
    "decision_neutral_logloss": {"features": "hydro-decision-v1",
                                 "decision_groups": ["economics", "water_balance"],
                                 "parameter_overrides": {"boost_from_average": False}},
    "decision_regularized_logloss": {"features": "hydro-decision-v1",
                                     "decision_groups": ["economics", "water_balance"],
                                     "parameter_overrides": {"num_leaves": 31, "max_depth": 6,
                                                             "min_child_samples": 500, "reg_lambda": 10.0}},
    "decision_slow_logloss": {"features": "hydro-decision-v1",
                              "decision_groups": ["economics", "water_balance"],
                              "parameter_overrides": {"learning_rate": 0.03}},
    "decision_logloss": {"features": "hydro-decision-v1",
                         "decision_groups": ["economics", "water_balance"]},
    "baseline_logloss": {"features": "hydro-context-v1", "decision_groups": []},
}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError("Unknown LightGBM experiment preset.")
FEATURE_VERSION = EXPERIMENTS[EXPERIMENT]["features"]
DECISION_GROUPS = set(EXPERIMENTS[EXPERIMENT]["decision_groups"])
FOLDS = [("2020-01-01", "2021-01-01"), ("2021-01-01", "2022-01-01")]
HOLDOUT_START = pd.Timestamp("2022-01-01", tz="UTC")
EARLY_STOPPING = 100
MAX_ITERATIONS = 2000
# Explicit starting values, not tuned values. Tune on development folds only.
BASE_PARAMETERS = {
    "objective": "binary", "boosting_type": "gbdt", "metric": "binary_logloss",
    "learning_rate": 0.05, "num_leaves": 63, "max_depth": 7,
    "min_child_samples": 200, "reg_lambda": 5.0, "reg_alpha": 0.0,
    "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
    "max_bin": 127, "device_type": "cpu", "deterministic": True,
    "force_col_wise": True, "verbosity": -1,
}
BASE_PARAMETERS.update(EXPERIMENTS[EXPERIMENT].get("parameter_overrides", {}))

# Keep the measured first LightGBM configuration as a reproducible reference.
LIGHTGBM_REFERENCE_CONFIG = {'model_family': 'lightgbm',
 'experiment': 'decision_logloss',
 'model_strategy': 'per_generator',
 'feature_version': 'hydro-decision-v1',
 'feature_signature': '413b34b792c90dc7fb586250679bd5438ef0dc4e90ef8b07a8bdb6331b6c2af9',
 'seed': 42,
 'parameters': {'objective': 'binary',
                'boosting_type': 'gbdt',
                'metric': 'binary_logloss',
                'learning_rate': 0.05,
                'num_leaves': 63,
                'max_depth': 7,
                'min_child_samples': 200,
                'reg_lambda': 5.0,
                'reg_alpha': 0.0,
                'subsample': 0.8,
                'subsample_freq': 1,
                'colsample_bytree': 0.8,
                'max_bin': 127,
                'device_type': 'cpu',
                'deterministic': True,
                'force_col_wise': True,
                'verbosity': -1},
 'iterations': {'Hogga_G1': 210,
                'Tokke_G1': 208,
                'Tokke_G2': 176,
                'Tokke_G3': 345,
                'Tokke_G4': 283,
                'Lio_G1': 110,
                'Byrte_G1': 443,
                'Vinje_G1': 162,
                'Vinje_G2': 165,
                'Vinje_G3': 286,
                'Haukeli_G1': 79,
                'Songa_G1': 196,
                'Kjela_G1': 358,
                'Vesle Kjela_G1': 367}}

# Frozen neutral-initialization configuration from completed assistant-run validation.
SELECTED_CONFIG = {'model_family': 'lightgbm',
 'experiment': 'decision_neutral_logloss',
 'model_strategy': 'per_generator',
 'feature_version': 'hydro-decision-v1',
 'feature_signature': '413b34b792c90dc7fb586250679bd5438ef0dc4e90ef8b07a8bdb6331b6c2af9',
 'seed': 42,
 'parameters': {'objective': 'binary',
                'boosting_type': 'gbdt',
                'metric': 'binary_logloss',
                'learning_rate': 0.05,
                'num_leaves': 63,
                'max_depth': 7,
                'min_child_samples': 200,
                'reg_lambda': 5.0,
                'reg_alpha': 0.0,
                'subsample': 0.8,
                'subsample_freq': 1,
                'colsample_bytree': 0.8,
                'max_bin': 127,
                'device_type': 'cpu',
                'deterministic': True,
                'force_col_wise': True,
                'verbosity': -1,
                'boost_from_average': False},
 'iterations': {'Hogga_G1': 257,
                'Tokke_G1': 235,
                'Tokke_G2': 184,
                'Tokke_G3': 321,
                'Tokke_G4': 256,
                'Lio_G1': 100,
                'Byrte_G1': 406,
                'Vinje_G1': 204,
                'Vinje_G2': 111,
                'Vinje_G3': 302,
                'Haukeli_G1': 69,
                'Songa_G1': 203,
                'Kjela_G1': 417,
                'Vesle Kjela_G1': 358}}

if RUN_MODE not in {"check", "smoke", "validate", "holdout", "submit"}:
    raise ValueError("Unknown RUN_MODE.")
if not (ROOT / "prediction_mapping.csv").is_file():
    raise FileNotFoundError("Run this notebook from the project root with the provided raw data present.")
if not 1 <= THREADS <= 4:
    raise ValueError("Use between 1 and 4 CPU threads for the course reproduction budget.")
NOTEBOOK_STARTED = perf_counter()
VERSIONS = {"python": platform.python_version()}
for package in ("numpy", "pandas", "PyYAML", "lightgbm", "scikit-learn", "matplotlib"):
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = "not installed"
print("Mode:", RUN_MODE, "Model:", MODEL_FAMILY, "Experiment:", EXPERIMENT, "Features:", FEATURE_VERSION)
print("Package versions:", VERSIONS)
if RUN_MODE != "check":
    missing = [p for p in ("lightgbm", "scikit-learn") if VERSIONS[p] == "not installed"]
    if missing:
        raise ImportError(f"Install requirements.txt into this kernel's environment yourself before training: {missing}")


In [ ]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")
display(pd.DataFrame({"generator": GENERATORS,
    "training_on_fraction": [float(labels_for(train_cases, g).mean()) for g in GENERATORS]}))

In [ ]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")
display(pd.DataFrame({"generator": GENERATORS,
                      "intake_reservoirs": [", ".join(FEEDERS[g]) for g in GENERATORS]}))
display(pd.DataFrame([{"input": k, "rows_in_task_range": len(v), "columns": len(v.columns),
                       "start": str(v.index.min()), "end": str(v.index.max())}
                      for k, v in tables.items()]))

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))


In [ ]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32, copy=False)

In [ ]:
def chronological_split(start, end):
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    if start >= end or end > HOLDOUT_START:
        raise ValueError("Development folds must end by the reserved holdout start.")
    horizon_end = train_cases["starttime"] + pd.Timedelta(hours=HOURS)
    fitting = train_cases.loc[horizon_end <= start].copy()
    validation = train_cases.loc[(train_cases["starttime"] >= start) & (horizon_end <= end)].copy()
    if fitting.empty or validation.empty:
        raise ValueError("Empty chronological fold.")
    if (fitting["starttime"] + pd.Timedelta(hours=HOURS)).max() > validation["starttime"].min():
        raise ValueError("Training and validation horizons overlap.")
    return fitting, validation

def even_sample(cases, maximum):
    indices = np.linspace(0, len(cases) - 1, min(maximum, len(cases)), dtype=int)
    return cases.iloc[indices].copy()

def assemble_predictions(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("Predictions must cover all 14 generators.")
    values = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for generator in GENERATORS:
        probability = np.asarray(predictions[generator])
        if probability.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {generator}")
        values[:, TARGET_POSITIONS[generator]] = probability
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    result = pd.DataFrame(values, columns=LABELS)
    result.insert(0, "Run No", cases["Run No"].to_numpy())
    return result

display(pd.DataFrame([{"validation_start": start, "validation_end_exclusive": end,
                       "training_cases": len(chronological_split(start, end)[0]),
                       "validation_cases": len(chronological_split(start, end)[1])}
                      for start, end in FOLDS]))
probe_cases = pd.concat([train_cases.iloc[[0, 1]], test_cases.iloc[[0, -1]]], ignore_index=True)
probe = feature_matrix(probe_cases)
probe_single = feature_matrix(probe_cases.iloc[[1]])
np.testing.assert_allclose(probe.iloc[HOURS:2 * HOURS].to_numpy(), probe_single.to_numpy())
for i, start in enumerate(probe_cases["starttime"]):
    rows = slice(i * HOURS, (i + 1) * HOURS)
    times = pd.date_range(start, periods=HOURS, freq="h")
    np.testing.assert_allclose(probe.iloc[rows]["price"], lookup("prices", times).reshape(-1))
    for reservoir in RESERVOIRS:
        np.testing.assert_allclose(probe.iloc[rows][f"initial_volume__{reservoir}"], tables["volumes"].at[start, reservoir])
        np.testing.assert_allclose(probe.iloc[rows][f"terminal_value__{reservoir}"], tables["water_values"].at[start + pd.Timedelta(hours=HOURS), reservoir])
for g in GENERATORS:
    generator_features(probe, g)
    truth = labels_for(train_cases.iloc[:2], g).reshape(2, HOURS)
    np.testing.assert_array_equal(truth[:, [0, 24, 167]], train_cases.iloc[:2][[TARGET_COLUMNS[g][h] for h in (0, 24, 167)]])
roundtrip = assemble_predictions(train_cases.iloc[:2],
    {g: labels_for(train_cases.iloc[:2], g).reshape(2, HOURS) for g in GENERATORS})
np.testing.assert_array_equal(roundtrip[LABELS], train_cases.iloc[:2][LABELS])
NUMERIC_FEATURE_NAMES = generator_features(probe, GENERATORS[0]).columns.tolist()
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + (["generator_id"] if MODEL_STRATEGY == "shared" else [])
MODEL_KEYS = ["shared"] if MODEL_STRATEGY == "shared" else GENERATORS
FEATURE_SIGNATURE = hashlib.sha256(json.dumps(FEATURE_NAMES).encode()).hexdigest()
if any(c in FEATURE_NAMES for c in ("Run No", "starttime", "total_calculation_time")):
    raise ValueError("Metadata leaked into predictors.")
bytes_per_case = HOURS * len(NUMERIC_FEATURE_NAMES) * np.dtype("float32").itemsize
print(f"Data checks passed: {len(FEATURE_NAMES)} features per generator-hour.")
print(f"One full training feature matrix: approximately {len(train_cases) * bytes_per_case / 2**20:.0f} MiB.")
print("This is not peak memory: copies, CatBoost pools, and model state add overhead.")
del probe, probe_single, roundtrip
if any(re.search(r'[\[\]{}":,]', name) for name in FEATURE_NAMES):
    raise ValueError("A feature name contains unsupported LightGBM JSON characters.")
print("LightGBM check mode audits preprocessing without fitting or importing LightGBM.")


In [ ]:
def safe_auc(actual, predicted):
    from sklearn.metrics import roc_auc_score
    actual = np.asarray(actual).reshape(-1)
    if np.unique(actual).size < 2:
        return None
    return float(roc_auc_score(actual, np.asarray(predicted).reshape(-1)))

def evaluate(cases, predictions):
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    frame = assemble_predictions(cases, predictions)
    actual = cases[LABELS].to_numpy(dtype=np.uint8)
    predicted = frame[LABELS].to_numpy()
    overall = float(roc_auc_score(actual, predicted, average="micro"))
    detail = pd.DataFrame([{"generator": g,
        "auc": safe_auc(labels_for(cases, g), predictions[g]),
        "logloss": float(log_loss(labels_for(cases, g), predictions[g].reshape(-1), labels=[0, 1])),
        "brier": float(brier_score_loss(labels_for(cases, g), predictions[g].reshape(-1))),
        "observed_on_fraction": float(labels_for(cases, g).mean()),
        "mean_prediction": float(predictions[g].mean())} for g in GENERATORS])
    return overall, detail, frame

def seasonal_predictions(fitting, validation):
    groups = fitting[LABELS].groupby(fitting["starttime"].dt.month).mean()
    if not set(validation["starttime"].dt.month).issubset(groups.index):
        raise ValueError("The seasonal reference lacks a validation month in training.")
    values = groups.loc[validation["starttime"].dt.month]
    return {g: values[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32) for g in GENERATORS}

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def fit_and_predict(fitting, evaluation, parameters, iterations, folder, *, early_stop):
    import lightgbm as lgb
    started = perf_counter()
    folder.mkdir(parents=True, exist_ok=False)
    X_fit_base, X_eval_base = feature_matrix(fitting), feature_matrix(evaluation)
    feature_seconds = perf_counter() - started
    predicted, records, importance, curves = {}, [], [], {}
    for generator in GENERATORS:
        generator_started = perf_counter()
        X_fit = generator_features(X_fit_base, generator)
        X_eval = generator_features(X_eval_base, generator)
        if X_fit.columns.tolist() != FEATURE_NAMES or X_eval.columns.tolist() != FEATURE_NAMES:
            raise ValueError("Feature schema changed between partitions.")
        y = labels_for(fitting, generator)
        limit = int(iterations[generator] if isinstance(iterations, dict) else iterations)
        if limit < 1:
            raise ValueError("Tree counts must be positive.")
        model = None
        if np.unique(y).size == 1:
            constant = float(y[0])
            probability = np.full(len(evaluation) * HOURS, constant, dtype=np.float32)
            tree_count = 1
            write_json(folder / f"{generator}_constant.json", {"probability": constant})
        else:
            constant = None
            model = lgb.LGBMClassifier(**parameters, n_estimators=limit,
                                       random_state=SEED, n_jobs=THREADS, importance_type="gain")
            history = {}
            callbacks = [lgb.log_evaluation(period=100), lgb.record_evaluation(history)]
            options = {"callbacks": callbacks}
            if early_stop:
                # Only the development target is read. Holdout/test labels are untouched.
                options.update(eval_set=[(X_eval, labels_for(evaluation, generator))],
                               eval_names=["validation"], eval_metric="binary_logloss")
                callbacks.append(lgb.early_stopping(EARLY_STOPPING, first_metric_only=True))
            print(f"Training {generator}: {len(y):,} hourly rows, at most {limit} trees", flush=True)
            model.fit(X_fit, y, **options)
            if list(model.classes_) != [0, 1]:
                raise ValueError("Unexpected LightGBM class order.")
            tree_count = int(model.best_iteration_ or model.booster_.current_iteration())
            if not 1 <= tree_count <= limit:
                raise ValueError("Invalid LightGBM retained tree count.")
            probability = model.predict_proba(X_eval, num_iteration=tree_count,
                                               validate_features=True)[:, 1].astype(np.float32)
            model.booster_.save_model(str(folder / f"{generator}.txt"), num_iteration=tree_count)
            curves[generator] = history
            gains = model.booster_.feature_importance(importance_type="gain", iteration=tree_count)
            normalized = 100 * gains / gains.sum() if gains.sum() > 0 else np.zeros_like(gains)
            importance.extend({"generator": generator, "feature": name,
                               "importance": float(value), "gain": float(gain)}
                              for name, value, gain in zip(FEATURE_NAMES, normalized, gains))
        predicted[generator] = probability.reshape(len(evaluation), HOURS)
        records.append({"generator": generator, "trees": tree_count,
                        "constant_probability": constant, "seconds": perf_counter() - generator_started})
        pd.DataFrame(records).to_csv(folder / "training_progress.csv", index=False)
        del X_fit, X_eval, model
        gc.collect()
    pd.DataFrame(importance, columns=["generator", "feature", "importance", "gain"]).to_csv(
        folder / "feature_importance.csv", index=False)
    write_json(folder / "learning_curves.json", curves)
    write_json(folder / "timing.json", {"feature_seconds": feature_seconds,
                                       "total_seconds": perf_counter() - started})
    del X_fit_base, X_eval_base
    gc.collect()
    return predicted, records, importance


In [ ]:
RUN_DIRECTORY = None
run_summary = None
importance_rows = []
if RUN_MODE == "check":
    print("Check mode complete. No models trained and no submission written.")
else:
    if RUN_MODE in {"holdout", "submit"}:
        if not isinstance(SELECTED_CONFIG, dict):
            raise ValueError("Paste a completed validate run's configuration into SELECTED_CONFIG first.")
        if SELECTED_CONFIG.get("model_family") != MODEL_FAMILY:
            raise ValueError("Use a completed LightGBM configuration, not a CatBoost configuration.")
        if SELECTED_CONFIG.get("feature_version") != FEATURE_VERSION or SELECTED_CONFIG.get("feature_signature") != FEATURE_SIGNATURE:
            raise ValueError("Frozen configuration and current features do not match.")
        if SELECTED_CONFIG.get("experiment", "baseline_logloss") != EXPERIMENT:
            raise ValueError("Set EXPERIMENT to the frozen configuration's experiment name before holdout or submit.")
        if SELECTED_CONFIG.get("seed") != SEED:
            raise ValueError("Use the same random seed as the frozen configuration.")
        if SELECTED_CONFIG.get("model_strategy", "per_generator") != MODEL_STRATEGY:
            raise ValueError("Frozen configuration uses a different model strategy.")
        if set(SELECTED_CONFIG.get("iterations", {})) != set(MODEL_KEYS):
            raise ValueError(f"Frozen configuration must contain tree counts for {MODEL_KEYS}.")
        if any(not isinstance(v, int) or v < 1 for v in SELECTED_CONFIG["iterations"].values()):
            raise ValueError("Frozen tree counts must be positive integers.")
        parameters = dict(SELECTED_CONFIG["parameters"])
        if parameters.get("device_type") != "cpu":
            raise ValueError("This notebook uses CPU LightGBM training.")
    else:
        parameters = dict(BASE_PARAMETERS)

    if set(parameters) & {"n_estimators", "num_iterations", "num_iteration", "num_boost_round",
                          "iterations", "random_state", "seed", "n_jobs", "num_threads",
                          "num_thread", "early_stopping_round", "early_stopping_rounds",
                          "early_stopping", "importance_type"}:
        raise ValueError("Keep tree budget, seed, threads and stopping outside the frozen parameter dictionary.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIRECTORY = ROOT / "outputs" / "lightgbm" / f"{stamp}_{RUN_MODE}_{EXPERIMENT}"
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    run_summary = {"status": "incomplete", "model_family": MODEL_FAMILY, "mode": RUN_MODE, "experiment": EXPERIMENT, "versions": VERSIONS,
        "seed": SEED, "threads": THREADS, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION,
        "feature_signature": FEATURE_SIGNATURE, "features": FEATURE_NAMES,
        "parameters": parameters, "folds": FOLDS,
        "maximum_iterations": MAX_ITERATIONS, "early_stopping_rounds": EARLY_STOPPING,
        "selected_configuration": SELECTED_CONFIG,
        "input_sha256": {k: file_hash(v) for k, v in RAW_FILES.items()}}
    write_json(RUN_DIRECTORY / "run.json", run_summary)

    if RUN_MODE in {"smoke", "validate"}:
        fold_results, iteration_records = [], []
        active_folds = FOLDS[-1:] if RUN_MODE == "smoke" else FOLDS
        for start, end in active_folds:
            fitting, validation = chronological_split(start, end)
            if RUN_MODE == "smoke":
                fitting, validation = even_sample(fitting, 96), even_sample(validation, 48)
            folder = RUN_DIRECTORY / f"fold_{start[:4]}"
            predicted, records, importance = fit_and_predict(
                fitting, validation, parameters, 40 if RUN_MODE == "smoke" else MAX_ITERATIONS,
                folder, early_stop=True)
            auc, detail, prediction_frame = evaluate(validation, predicted)
            baseline_auc, _, _ = evaluate(validation, seasonal_predictions(fitting, validation))
            detail.to_csv(folder / "per_generator_metrics.csv", index=False)
            prediction_frame.to_csv(folder / "validation_predictions.csv", index=False)
            iteration_records.extend(records)
            importance_rows.extend(dict(row, fold=start[:4]) for row in importance)
            fold_results.append({"validation_start": start, "validation_end_exclusive": end,
                                 "training_cases": len(fitting), "validation_cases": len(validation),
                                 "micro_auc": auc, "seasonal_micro_auc": baseline_auc,
                                 "mean_generator_logloss": float(detail["logloss"].mean()),
                                 "mean_generator_brier": float(detail["brier"].mean())})
            print(f"{start[:4]} micro ROC-AUC: {auc:.6f}; same-fold seasonal reference: {baseline_auc:.6f}")
            display(detail)
            del predicted, prediction_frame
        run_summary["development_results"] = fold_results
        display(pd.DataFrame(fold_results))
        if RUN_MODE == "validate":
            selected = {"model_family": MODEL_FAMILY, "experiment": EXPERIMENT, "model_strategy": MODEL_STRATEGY, "feature_version": FEATURE_VERSION, "feature_signature": FEATURE_SIGNATURE,
                        "seed": SEED, "parameters": parameters,
                        "iterations": {g: max(1, int(np.median([r["trees"] for r in iteration_records if r["generator"] == g]))) for g in MODEL_KEYS}}
            write_json(RUN_DIRECTORY / "selected_configuration.json", selected)
            print("Copy this complete dictionary into SELECTED_CONFIG before holdout or submit:")
            print(repr(selected))
            if any(r["trees"] >= MAX_ITERATIONS for r in iteration_records):
                print("Some models reached MAX_ITERATIONS. Inspect their learning curves before freezing the budget.")
    elif RUN_MODE == "holdout":
        fitting = train_cases.loc[train_cases["starttime"] + pd.Timedelta(hours=HOURS) <= HOLDOUT_START]
        holdout = train_cases.loc[train_cases["starttime"] >= HOLDOUT_START]
        if fitting.empty or holdout.empty:
            raise ValueError("Empty holdout partition.")
        predicted, _, importance_rows = fit_and_predict(fitting, holdout, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "holdout", early_stop=False)
        auc, detail, prediction_frame = evaluate(holdout, predicted)
        prediction_frame.to_csv(RUN_DIRECTORY / "holdout_predictions.csv", index=False)
        detail.to_csv(RUN_DIRECTORY / "holdout_metrics.csv", index=False)
        run_summary["holdout_micro_auc"] = auc
        print(f"Frozen-configuration holdout micro ROC-AUC: {auc:.6f}")
        display(detail)
    elif RUN_MODE == "submit":
        predicted, _, importance_rows = fit_and_predict(train_cases, test_cases, parameters,
            SELECTED_CONFIG["iterations"], RUN_DIRECTORY / "full_training", early_stop=False)
        submission = assemble_predictions(test_cases, predicted)
        if not submission.columns.equals(template.columns) or not submission["Run No"].equals(template["Run No"]):
            raise ValueError("Submission IDs or column order differ from the template.")
        if submission.shape != template.shape:
            raise ValueError("Submission shape differs from the template.")
        destination = RUN_DIRECTORY / "lightgbm_submission.csv"
        submission.to_csv(destination, index=False)
        reloaded = pd.read_csv(destination)
        np.testing.assert_allclose(reloaded[LABELS], submission[LABELS], rtol=1e-6, atol=1e-8)
        np.testing.assert_array_equal(reloaded["Run No"], template["Run No"])
        run_summary["submission"] = str(destination.relative_to(ROOT))
        print(f"Verified submission saved: {destination}")

    run_summary["status"] = "complete"
    run_summary["total_seconds"] = perf_counter() - NOTEBOOK_STARTED
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    (RUN_DIRECTORY / "environment.txt").write_text(
        "\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n", encoding="utf-8")
    print(f"Recorded runtime so far: {run_summary['total_seconds'] / 60:.1f} minutes")
    print("Run artifacts:", RUN_DIRECTORY)

In [ ]:

if ENSEMBLE_MODE == "submit":
    if run_summary["status"] != "complete" or run_summary["mode"] != "submit":
        raise ValueError("Component did not complete a full-data refit.")
    COMPONENT_METADATA['lightgbm'] = run_summary
    COMPONENT_FRAMES['lightgbm'] = pd.read_csv(run_summary["submission"])


## Tcn component

Frozen experiment: `tcn_wide`. The following pipeline is inlined.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import platform
import random
import re
import copy

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd()
RUN_MODE = ENSEMBLE_MODE
EXPERIMENT = 'tcn_wide'
MODEL_FAMILY = "tcn"
SEED = 42
THREADS = 4
HOURS = 168
FOLDS = [("2020-01-01", "2021-01-01"), ("2021-01-01", "2022-01-01")]
HOLDOUT_START = pd.Timestamp("2022-01-01", tz="UTC")
# These settings identify formulas copied from the verified tree pipeline.
FEATURE_VERSION = "hydro-decision-v1"
DECISION_GROUPS = {"economics", "water_balance"}
EXPERIMENTS = {
    "tcn_context": {"width": 64, "dropout": 0.10},
    "tcn_regularized": {"width": 64, "dropout": 0.25, "weight_decay": 0.001},
    "tcn_wide": {"width": 96, "dropout": 0.10},
    "tcn_topology": {"width": 64, "dropout": 0.10, "topology_context": True},
}
BASE_PARAMETERS = {"width": 64, "dropout": 0.10,
    "dilations": [1, 2, 4, 8, 16, 32, 64], "kernel_size": 3,
    "batch_size": 32, "learning_rate": 0.001, "weight_decay": 0.0001,
    "max_epochs": 100, "patience": 15, "gradient_clip": 1.0,
    "normalization_clip": 10.0, "selection_metric": "micro_auc",
    "lr_schedule": "cosine_epoch_max_epochs", "device": "cpu"}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError("Unknown TCN experiment.")
BASE_PARAMETERS.update(EXPERIMENTS[EXPERIMENT])
SEQUENCE_FEATURE_VERSION = ("tcn-intake-topology-v2" if BASE_PARAMETERS.get("topology_context", False)
                            else "tcn-intake-context-v1")
SELECTED_CONFIG = {'model_family': 'tcn',
 'experiment': 'tcn_wide',
 'feature_version': 'tcn-intake-context-v1',
 'feature_signature': '0b40966bc860b876ada3f82558fae5198d719628e4d9acf94c2bdc08474f31af',
 'seed': 42,
 'parameters': {'width': 96,
                'dropout': 0.1,
                'dilations': [1, 2, 4, 8, 16, 32, 64],
                'kernel_size': 3,
                'batch_size': 32,
                'learning_rate': 0.001,
                'weight_decay': 0.0001,
                'max_epochs': 100,
                'patience': 15,
                'gradient_clip': 1.0,
                'normalization_clip': 10.0,
                'selection_metric': 'micro_auc',
                'lr_schedule': 'cosine_epoch_max_epochs',
                'device': 'cpu'},
 'epochs': 10}
# Optional UTC deadline provided by the development runner. Standalone runs have none.
DEADLINE_UTC = None
INTERPRET = True
if RUN_MODE not in {"check", "smoke", "validate", "holdout", "submit"}:
    raise ValueError("Unknown RUN_MODE.")
if not 1 <= THREADS <= 4:
    raise ValueError("Use one to four CPU threads.")
NOTEBOOK_STARTED = perf_counter()
VERSIONS = {"python": platform.python_version()}
for package in ("numpy", "pandas", "PyYAML", "torch", "scikit-learn"):
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = "not installed"
if RUN_MODE != "check" and VERSIONS["torch"] == "not installed":
    raise ImportError("Install PyTorch into the notebook environment before training.")
print("Mode:", RUN_MODE, "Experiment:", EXPERIMENT, "Versions:", VERSIONS)

In [ ]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")
display(pd.DataFrame({"generator": GENERATORS,
    "training_on_fraction": [float(labels_for(train_cases, g).mean()) for g in GENERATORS]}))

In [ ]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")
display(pd.DataFrame({"generator": GENERATORS,
                      "intake_reservoirs": [", ".join(FEEDERS[g]) for g in GENERATORS]}))
display(pd.DataFrame([{"input": k, "rows_in_task_range": len(v), "columns": len(v.columns),
                       "start": str(v.index.min()), "end": str(v.index.max())}
                      for k, v in tables.items()]))

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))

In [ ]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32, copy=False)

In [ ]:
# Retain the verified chronological split and output mapping, with no model fitting.
def chronological_split(start, end):
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    if start >= end or end > HOLDOUT_START:
        raise ValueError("Development folds must end by the reserved holdout start.")
    horizon_end = train_cases["starttime"] + pd.Timedelta(hours=HOURS)
    fitting = train_cases.loc[horizon_end <= start].copy()
    validation = train_cases.loc[(train_cases["starttime"] >= start) & (horizon_end <= end)].copy()
    if fitting.empty or validation.empty:
        raise ValueError("Empty chronological fold.")
    if (fitting["starttime"] + pd.Timedelta(hours=HOURS)).max() > validation["starttime"].min():
        raise ValueError("Training and validation horizons overlap.")
    return fitting, validation

def even_sample(cases, maximum):
    indices = np.linspace(0, len(cases) - 1, min(maximum, len(cases)), dtype=int)
    return cases.iloc[indices].copy()

def assemble_predictions(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("Predictions must cover all 14 generators.")
    values = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for generator in GENERATORS:
        probability = np.asarray(predictions[generator])
        if probability.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {generator}")
        values[:, TARGET_POSITIONS[generator]] = probability
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    result = pd.DataFrame(values, columns=LABELS)
    result.insert(0, "Run No", cases["Run No"].to_numpy())
    return result



INTAKE_GROUPS = list(dict.fromkeys(tuple(FEEDERS[g]) for g in GENERATORS))
GROUP_REPRESENTATIVES = [next(g for g in GENERATORS if tuple(FEEDERS[g]) == intake)
                         for intake in INTAKE_GROUPS]
STATIC_PREFIXES = ("initial_volume__", "initial_fraction__", "terminal_value__",
                   "inflow_week_mean__", "min_flow_week_max__", "min_volume_week_max__",
                   "price_week_mean", "price_week_std", "price_week_min", "price_week_max")
LOCAL_STATIC = {"local_initial_fraction", "local_terminal_value_capacity_weighted",
                "local_available_water_proxy_fraction", "water_end_no_release_excess_fraction"}

def sequence_inputs(cases):
    base = feature_matrix(cases)
    n = len(cases)
    dynamic_names = [name for name in base if not name.startswith(STATIC_PREFIXES)]
    static_names = [name for name in base if name.startswith(STATIC_PREFIXES)]
    dynamic = [base[dynamic_names].to_numpy().reshape(n, HOURS, -1)]
    static = [base[static_names].to_numpy().reshape(n, HOURS, -1)[:, 0]]
    # Repeated calendar coordinates are more naturally encoded on a circle for a NN.
    h = np.arange(HOURS)[None, :]
    times = pd.DatetimeIndex(np.repeat(cases.starttime.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    cycles = np.stack([np.sin(2*np.pi*times.hour/24), np.cos(2*np.pi*times.hour/24),
                       np.sin(2*np.pi*times.dayofweek/7), np.cos(2*np.pi*times.dayofweek/7)], axis=1)
    dynamic.append(cycles.reshape(n, HOURS, 4).astype(np.float32))
    dynamic_names += ["hour_sin", "hour_cos", "weekday_sin", "weekday_cos"]
    for group_index, generator in enumerate(GROUP_REPRESENTATIVES):
        extended = generator_features(base, generator)
        local = [name for name in extended if name not in base]
        local_dynamic = [name for name in local if name not in LOCAL_STATIC]
        local_static = [name for name in local if name in LOCAL_STATIC]
        dynamic.append(extended[local_dynamic].to_numpy().reshape(n, HOURS, -1))
        static.append(extended[local_static].to_numpy().reshape(n, HOURS, -1)[:, 0])
        dynamic_names += [f"intake_{group_index}::{name}" for name in local_dynamic]
        static_names += [f"intake_{group_index}::{name}" for name in local_static]
        del extended
    if BASE_PARAMETERS.get("topology_context", False):
        # Static initial levels only: no weekly realized storage or routed-flow forecast.
        levels = {}
        for reservoir in RESERVOIRS:
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            x, y = np.asarray(curve["x"], dtype=float), np.asarray(curve["y"], dtype=float)
            if len(x) != len(y) or not np.all(np.diff(x) > 0):
                raise ValueError("Invalid supplied volume-level curve.")
            levels[reservoir] = np.interp(base[f"initial_volume__{reservoir}"].to_numpy().reshape(n, HOURS)[:, 0], x, y)
            static.append(levels[reservoir][:, None].astype(np.float32))
            static_names.append(f"initial_level_m::{reservoir}")
        for group_index, generator in enumerate(GROUP_REPRESENTATIVES):
            intake = FEEDERS[generator]
            capacity = sum(CAPACITY[r] for r in intake)
            head = sum(levels[r]*CAPACITY[r] for r in intake)/capacity - OUTLET_ELEVATION[generator]
            static.append(head[:, None].astype(np.float32))
            static_names.append(f"intake_{group_index}::initial_gross_head_proxy_m")
            upstream = UPSTREAM[generator]
            upstream_capacity = sum(CAPACITY[r] for r in upstream)
            upstream_initial = sum(base[f"initial_volume__{r}"].to_numpy().reshape(n, HOURS)[:, 0] for r in upstream)
            upstream_value = sum(base[f"terminal_value__{r}"].to_numpy().reshape(n, HOURS)[:, 0]*CAPACITY[r] for r in upstream)/upstream_capacity
            static.append(np.stack([upstream_initial/upstream_capacity, upstream_value], axis=1).astype(np.float32))
            static_names += [f"intake_{group_index}::upstream_initial_fraction", f"intake_{group_index}::upstream_terminal_value"]
            upstream_flows = np.stack([sum(base[f"{prefix}__{r}"].to_numpy().reshape(n, HOURS) for r in upstream)
                               for prefix in ("inflow", "inflow_cumulative_Mm3", "inflow_remaining_Mm3")], axis=2)
            upstream_flows[:, :, 1:] /= upstream_capacity
            dynamic.append(upstream_flows.astype(np.float32))
            dynamic_names += [f"intake_{group_index}::upstream_natural_inflow",
                              f"intake_{group_index}::upstream_cumulative_inflow_fraction",
                              f"intake_{group_index}::upstream_remaining_inflow_fraction"]
    dynamic = np.ascontiguousarray(np.concatenate(dynamic, axis=2), dtype=np.float32)
    static = np.ascontiguousarray(np.concatenate(static, axis=1), dtype=np.float32)
    if not np.isfinite(dynamic).all() or not np.isfinite(static).all():
        raise ValueError("Sequence inputs must be finite.")
    return dynamic, static, dynamic_names, static_names

def sequence_labels(cases):
    return np.stack([cases[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32)
                     for g in GENERATORS], axis=2)

def sequence_frame(cases, probabilities):
    values = np.asarray(probabilities)
    if values.shape != (len(cases), HOURS, len(GENERATORS)):
        raise ValueError("Expected (cases, 168, 14) probabilities.")
    return assemble_predictions(cases, {g: values[:, :, j] for j, g in enumerate(GENERATORS)})

probe_cases = pd.concat([train_cases.iloc[[0, 21]], test_cases.iloc[[0, -1]]], ignore_index=True)
probe_dynamic, probe_static, DYNAMIC_NAMES, STATIC_NAMES = sequence_inputs(probe_cases)
single_dynamic, single_static, _, _ = sequence_inputs(probe_cases.iloc[[1]])
np.testing.assert_array_equal(probe_dynamic[1:2], single_dynamic)
np.testing.assert_array_equal(probe_static[1:2], single_static)
np.testing.assert_array_equal(sequence_frame(train_cases.iloc[:2], sequence_labels(train_cases.iloc[:2]))[LABELS],
                              train_cases.iloc[:2][LABELS])
FEATURE_NAMES = {"dynamic": DYNAMIC_NAMES, "static": STATIC_NAMES,
                 "intakes": [list(group) for group in INTAKE_GROUPS], "outputs": GENERATORS}
FEATURE_SIGNATURE = hashlib.sha256(json.dumps(FEATURE_NAMES).encode()).hexdigest()
for start, end in FOLDS:
    fitting, validation = chronological_split(start, end)
    print(start, "fit/validation:", len(fitting), len(validation))
print("Sequence schema:", len(DYNAMIC_NAMES), "hourly channels;", len(STATIC_NAMES), "static channels")
print("Input bytes per case:", probe_dynamic[0].nbytes + probe_static[0].nbytes)
del probe_dynamic, probe_static, single_dynamic, single_static

In [ ]:
def fit_scaler(dynamic, static, clip):
    return {"dynamic_mean": dynamic.mean(axis=(0, 1), dtype=np.float64).astype(np.float32),
            "dynamic_std": np.maximum(dynamic.std(axis=(0, 1), dtype=np.float64), 1e-6).astype(np.float32),
            "static_mean": static.mean(axis=0, dtype=np.float64).astype(np.float32),
            "static_std": np.maximum(static.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32),
            "clip": float(clip)}

def scale_inputs(dynamic, static, scaler):
    d = np.clip((dynamic - scaler["dynamic_mean"]) / scaler["dynamic_std"], -scaler["clip"], scaler["clip"])
    s = np.clip((static - scaler["static_mean"]) / scaler["static_std"], -scaler["clip"], scaler["clip"])
    return np.ascontiguousarray(d, dtype=np.float32), np.ascontiguousarray(s, dtype=np.float32)

def initialize_torch():
    import torch
    torch.set_num_threads(THREADS)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    return torch

def make_model(parameters):
    import torch
    from torch import nn
    class ChannelNorm(nn.Module):
        def __init__(self, width):
            super().__init__()
            self.norm = nn.LayerNorm(width)
        def forward(self, x):
            return self.norm(x.transpose(1, 2)).transpose(1, 2)
    class ResidualBlock(nn.Module):
        def __init__(self, width, dilation):
            super().__init__()
            layers = []
            for _ in range(2):
                layers += [nn.Conv1d(width, width, parameters["kernel_size"],
                                    dilation=dilation, padding=dilation),
                           ChannelNorm(width), nn.GELU(), nn.Dropout(parameters["dropout"])]
            self.layers = nn.Sequential(*layers)
        def forward(self, x):
            return x + self.layers(x)
    class HydroTCN(nn.Module):
        def __init__(self):
            super().__init__()
            width = parameters["width"]
            self.projection = nn.Conv1d(len(DYNAMIC_NAMES), width, 1)
            self.condition = nn.Sequential(nn.Linear(len(STATIC_NAMES) + len(DYNAMIC_NAMES), width),
                                           nn.GELU(), nn.Linear(width, width))
            self.blocks = nn.Sequential(*[ResidualBlock(width, dilation) for dilation in parameters["dilations"]])
            self.head = nn.Sequential(ChannelNorm(width), nn.GELU(), nn.Conv1d(width, len(GENERATORS), 1))
            self.linear_skip = nn.Conv1d(len(DYNAMIC_NAMES), len(GENERATORS), 1)
            self.static_skip = nn.Linear(len(STATIC_NAMES), len(GENERATORS))
        def forward(self, dynamic, static):
            sequence = dynamic.transpose(1, 2)
            context = self.condition(torch.cat([static, dynamic.mean(dim=1)], dim=1)).unsqueeze(2)
            hidden = self.blocks(self.projection(sequence) + context)
            logits = self.head(hidden) + self.linear_skip(sequence) + self.static_skip(static).unsqueeze(2)
            return logits.transpose(1, 2)
    if parameters["kernel_size"] != 3 or parameters["device"] != "cpu":
        raise ValueError("This verified architecture uses kernel size 3 on CPU.")
    return HydroTCN()

def check_deadline():
    if DEADLINE_UTC is not None and datetime.now(timezone.utc) >= datetime.fromisoformat(DEADLINE_UTC):
        raise TimeoutError("The authorized experiment deadline has been reached.")

def predict_sequence(model, dynamic, static, batch_size):
    import torch
    model.eval()
    chunks = []
    with torch.inference_mode():
        for start in range(0, len(dynamic), batch_size):
            check_deadline()
            logits = model(torch.from_numpy(dynamic[start:start+batch_size]),
                           torch.from_numpy(static[start:start+batch_size]))
            chunks.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(chunks).astype(np.float32)

In [ ]:
def safe_auc(actual, predicted):
    from sklearn.metrics import roc_auc_score
    actual = np.asarray(actual).reshape(-1)
    if np.unique(actual).size < 2:
        return None
    return float(roc_auc_score(actual, np.asarray(predicted).reshape(-1)))

def evaluate(cases, predictions):
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    frame = assemble_predictions(cases, predictions)
    actual = cases[LABELS].to_numpy(dtype=np.uint8)
    predicted = frame[LABELS].to_numpy()
    overall = float(roc_auc_score(actual, predicted, average="micro"))
    detail = pd.DataFrame([{"generator": g,
        "auc": safe_auc(labels_for(cases, g), predictions[g]),
        "logloss": float(log_loss(labels_for(cases, g), predictions[g].reshape(-1), labels=[0, 1])),
        "brier": float(brier_score_loss(labels_for(cases, g), predictions[g].reshape(-1))),
        "observed_on_fraction": float(labels_for(cases, g).mean()),
        "mean_prediction": float(predictions[g].mean())} for g in GENERATORS])
    return overall, detail, frame

def seasonal_predictions(fitting, validation):
    groups = fitting[LABELS].groupby(fitting["starttime"].dt.month).mean()
    if not set(validation["starttime"].dt.month).issubset(groups.index):
        raise ValueError("The seasonal reference lacks a validation month in training.")
    values = groups.loc[validation["starttime"].dt.month]
    return {g: values[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32) for g in GENERATORS}

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()



def sensitivity_table(model, dynamic, static, folder):
    # Gradients are local associations in standardized units, not causal importance.
    import torch
    model.eval()
    rows = []
    for j, generator in enumerate(GENERATORS):
        check_deadline()
        d = torch.tensor(dynamic[:8], requires_grad=True)
        s = torch.tensor(static[:8], requires_grad=True)
        output = model(d, s)[:, :, j].mean()
        gd, gs = torch.autograd.grad(output, (d, s))
        for group, names, values in (("dynamic", DYNAMIC_NAMES, gd.abs().mean((0, 1)).numpy()),
                                     ("static", STATIC_NAMES, gs.abs().mean(0).numpy())):
            rows.extend({"generator": generator, "group": group, "feature": name,
                         "mean_absolute_logit_gradient": float(value)} for name, value in zip(names, values))
    pd.DataFrame(rows).to_csv(folder / "input_sensitivity.csv", index=False)

def fit_and_predict(fitting, evaluation, parameters, epochs, folder, early_stop):
    torch = initialize_torch()
    from torch import nn
    started = perf_counter()
    folder.mkdir(parents=True, exist_ok=False)
    check_deadline()
    raw_d, raw_s, dn, sn = sequence_inputs(fitting)
    eval_d, eval_s, edn, esn = sequence_inputs(evaluation)
    if dn != DYNAMIC_NAMES or sn != STATIC_NAMES or edn != dn or esn != sn:
        raise ValueError("Feature schema changed between partitions.")
    scaler = fit_scaler(raw_d, raw_s, parameters["normalization_clip"])
    fit_d, fit_s = scale_inputs(raw_d, raw_s, scaler)
    val_d, val_s = scale_inputs(eval_d, eval_s, scaler)
    del raw_d, raw_s, eval_d, eval_s
    np.savez(folder / "scaler.npz", **scaler)
    y = torch.from_numpy(sequence_labels(fitting))
    actual = evaluation[LABELS].to_numpy(dtype=np.uint8) if early_stop else None
    model = make_model(parameters)
    optimizer = torch.optim.AdamW(model.parameters(), lr=parameters["learning_rate"],
                                  weight_decay=parameters["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()
    best_score, best_epoch, best_state, stale = -np.inf, 0, None, 0
    history = []
    parameter_count = sum(p.numel() for p in model.parameters())
    print(f"Training {parameter_count:,} parameters on {len(fitting)} whole cases; maximum {epochs} epochs", flush=True)
    preparation_seconds = perf_counter() - started
    for epoch in range(1, int(epochs)+1):
        check_deadline()
        epoch_started = perf_counter()
        # Keep this denominator frozen when refitting for a selected number of epochs.
        fraction = (epoch - 1) / max(parameters["max_epochs"]-1, 1)
        lr = parameters["learning_rate"] * (0.1 + 0.9 * (1 + np.cos(np.pi * fraction)) / 2)
        for group in optimizer.param_groups:
            group["lr"] = float(lr)
        model.train()
        order = torch.randperm(len(fitting)).numpy()
        accumulated_loss = 0.0
        for start in range(0, len(order), parameters["batch_size"]):
            check_deadline()
            indices = order[start:start+parameters["batch_size"]]
            optimizer.zero_grad(set_to_none=True)
            logits = model(torch.from_numpy(fit_d[indices]), torch.from_numpy(fit_s[indices]))
            loss = criterion(logits, y[indices])
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), parameters["gradient_clip"])
            optimizer.step()
            accumulated_loss += float(loss.detach()) * len(indices)
        record = {"epoch": epoch, "learning_rate": float(lr), "training_logloss": accumulated_loss/len(fitting)}
        if early_stop:
            probability = predict_sequence(model, val_d, val_s, parameters["batch_size"])
            frame = sequence_frame(evaluation, probability)
            from sklearn.metrics import roc_auc_score, log_loss
            score = float(roc_auc_score(actual, frame[LABELS].to_numpy(), average="micro"))
            record.update(micro_auc=score,
                validation_logloss=float(log_loss(actual.reshape(-1), frame[LABELS].to_numpy().reshape(-1), labels=[0, 1])))
            if score > best_score + 1e-7:
                best_score, best_epoch, stale = score, epoch, 0
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, folder / "best_state.pt")
            else:
                stale += 1
        record["seconds"] = perf_counter() - epoch_started
        history.append(record)
        pd.DataFrame(history).to_csv(folder / "learning_curve.csv", index=False)
        print(f"Epoch {epoch}: loss={record['training_logloss']:.5f}, "
              f"AUC={record.get('micro_auc', 'refit')}, {record['seconds']:.1f}s", flush=True)
        if early_stop and stale >= parameters["patience"]:
            break
    if early_stop:
        model.load_state_dict(best_state)
    else:
        best_epoch = int(epochs)
    torch.save(model.state_dict(), folder / "model_state.pt")
    probability = predict_sequence(model, val_d, val_s, parameters["batch_size"])
    if INTERPRET:
        sensitivity_table(model, val_d, val_s, folder)
    result = {"epochs": best_epoch, "epochs_run": len(history), "parameters_count": parameter_count,
              "preparation_seconds": preparation_seconds, "seconds": perf_counter() - started}
    write_json(folder / "training.json", result)
    del model, optimizer, fit_d, fit_s, val_d, val_s, y
    gc.collect()
    return {g: probability[:, :, j] for j, g in enumerate(GENERATORS)}, result

def frozen_configuration(results):
    return {"model_family": MODEL_FAMILY, "experiment": EXPERIMENT,
            "feature_version": SEQUENCE_FEATURE_VERSION, "feature_signature": FEATURE_SIGNATURE,
            "seed": SEED, "parameters": BASE_PARAMETERS,
            "epochs": int(np.median([r["selected_epochs"] for r in results]))}

def validate_frozen(configuration):
    if not isinstance(configuration, dict):
        raise ValueError("Copy a completed validation SELECTED_CONFIG before holdout or submit.")
    expected = {"model_family": MODEL_FAMILY, "experiment": EXPERIMENT,
                "feature_version": SEQUENCE_FEATURE_VERSION, "feature_signature": FEATURE_SIGNATURE,
                "seed": SEED, "parameters": BASE_PARAMETERS}
    if any(configuration.get(key) != value for key, value in expected.items()):
        raise ValueError("Frozen TCN configuration does not match current inputs, architecture or seed.")
    if not isinstance(configuration.get("epochs"), int) or not 1 <= configuration["epochs"] <= BASE_PARAMETERS["max_epochs"]:
        raise ValueError("Frozen epoch count must be within the validated training budget.")

In [ ]:
RUN_DIRECTORY = None
run_summary = None
if RUN_MODE == "check":
    print("Check mode complete. No training or submission generation.")
else:
    if RUN_MODE in {"holdout", "submit"}:
        validate_frozen(SELECTED_CONFIG)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIRECTORY = ROOT / "outputs/tcn" / f"{stamp}_{RUN_MODE}_{EXPERIMENT}"
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
    run_summary = {"status": "running", "mode": RUN_MODE, "model_family": MODEL_FAMILY,
        "experiment": EXPERIMENT, "feature_version": SEQUENCE_FEATURE_VERSION,
        "feature_signature": FEATURE_SIGNATURE, "seed": SEED, "threads": THREADS,
        "parameters": BASE_PARAMETERS, "folds": FOLDS,
        "input_sha256": {key: file_hash(path) for key, path in RAW_FILES.items()},
        "versions": VERSIONS, "feature_names": FEATURE_NAMES, "development_results": []}
    write_json(RUN_DIRECTORY / "run.json", run_summary)
    (RUN_DIRECTORY / "environment.txt").write_text("\n".join(f"{key}=={value}" for key, value in VERSIONS.items()), encoding="utf-8")
    try:
        if RUN_MODE in {"smoke", "validate"}:
            folds = FOLDS[:1] if RUN_MODE == "smoke" else FOLDS
            for start, end in folds:
                fitting, validation = chronological_split(start, end)
                if RUN_MODE == "smoke":
                    fitting, validation = even_sample(fitting, 96), even_sample(validation, 32)
                folder = RUN_DIRECTORY / f"fold_{start[:4]}"
                probability, training = fit_and_predict(fitting, validation, BASE_PARAMETERS,
                    3 if RUN_MODE == "smoke" else BASE_PARAMETERS["max_epochs"], folder, early_stop=True)
                auc, detail, frame = evaluate(validation, probability)
                detail.to_csv(folder / "per_generator_metrics.csv", index=False)
                frame.to_csv(folder / "validation_predictions.csv", index=False)
                baseline, _, _ = evaluate(validation, seasonal_predictions(fitting, validation))
                result = {"validation_start": start, "validation_end_exclusive": end,
                    "training_cases": len(fitting), "validation_cases": len(validation),
                    "micro_auc": auc, "seasonal_micro_auc": baseline,
                    "selected_epochs": training["epochs"], "seconds": training["seconds"]}
                run_summary["development_results"].append(result)
                write_json(RUN_DIRECTORY / "run.json", run_summary)
                print("Development fold:", result)
            if RUN_MODE == "validate":
                selected = frozen_configuration(run_summary["development_results"])
                write_json(RUN_DIRECTORY / "selected_configuration.json", selected)
                print("SELECTED_CONFIG =", repr(selected))
        else:
            if RUN_MODE == "holdout":
                fitting = train_cases.loc[train_cases.starttime + pd.Timedelta(hours=HOURS) <= HOLDOUT_START].copy()
                evaluation = train_cases.loc[train_cases.starttime >= HOLDOUT_START].copy()
            else:
                fitting, evaluation = train_cases, test_cases
            probability, training = fit_and_predict(fitting, evaluation, BASE_PARAMETERS,
                SELECTED_CONFIG["epochs"], RUN_DIRECTORY / "full_fit", early_stop=False)
            run_summary["selected_configuration"] = SELECTED_CONFIG
            run_summary["training"] = training
            if RUN_MODE == "holdout":
                auc, detail, frame = evaluate(evaluation, probability)
                run_summary["holdout_micro_auc"] = auc
                detail.to_csv(RUN_DIRECTORY / "holdout_per_generator_metrics.csv", index=False)
                frame.to_csv(RUN_DIRECTORY / "holdout_predictions.csv", index=False)
                print("Frozen-configuration 2022 micro ROC-AUC:", auc)
            else:
                frame = assemble_predictions(evaluation, probability)
                if frame.shape != template.shape or not frame.columns.equals(template.columns):
                    raise ValueError("Submission schema differs from the template.")
                np.testing.assert_array_equal(frame["Run No"], template["Run No"])
                submission = RUN_DIRECTORY / "tcn_submission.csv"
                frame.to_csv(submission, index=False)
                checked = pd.read_csv(submission)
                np.testing.assert_allclose(checked[LABELS], frame[LABELS], atol=1e-7, rtol=1e-7)
                np.testing.assert_array_equal(checked["Run No"], template["Run No"])
                run_summary["submission"] = str(submission)
                print("Verified submission:", submission, frame.shape)
        run_summary.update(status="complete", total_seconds=perf_counter() - NOTEBOOK_STARTED)
    except Exception as error:
        run_summary.update(status="failed", error=str(error), total_seconds=perf_counter() - NOTEBOOK_STARTED)
        raise
    finally:
        write_json(RUN_DIRECTORY / "run.json", run_summary)
    print("Run directory:", RUN_DIRECTORY)
    print("Total seconds:", run_summary["total_seconds"])

In [ ]:

if ENSEMBLE_MODE == "submit":
    if run_summary["status"] != "complete" or run_summary["mode"] != "submit":
        raise ValueError("Component did not complete a full-data refit.")
    COMPONENT_METADATA['tcn'] = run_summary
    COMPONENT_FRAMES['tcn'] = pd.read_csv(run_summary["submission"])


## Verified probability blend

Each generated component must match the template's exact IDs, columns and order.
Raw-input hashes are checked across components; feature signatures may differ
between trees and TCN by design. Probabilities are blended without thresholds,
generator-wise ranking or postprocessing that changes their cross-unit scale.


In [ ]:

if ENSEMBLE_MODE == "check":
    print("All selected component input audits passed. No models fitted or CSV generated.")
else:
    template = pd.read_csv("sample_submission.csv")
    if set(COMPONENT_FRAMES) != {name for name, weight in ENSEMBLE_WEIGHTS.items() if weight > 0}:
        raise ValueError("Missing selected component output.")
    reference_hashes = next(iter(COMPONENT_METADATA.values()))["input_sha256"]
    probabilities = np.zeros((len(template), len(template.columns)-1), dtype=np.float64)
    for name, frame in COMPONENT_FRAMES.items():
        if frame.shape != template.shape or not frame.columns.equals(template.columns):
            raise ValueError(f"Invalid component schema: {name}")
        np.testing.assert_array_equal(frame["Run No"], template["Run No"])
        values = frame.iloc[:, 1:].to_numpy(dtype=np.float64)
        if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
            raise ValueError(f"Invalid component probabilities: {name}")
        if COMPONENT_METADATA[name]["input_sha256"] != reference_hashes:
            raise ValueError("Component raw-input hashes differ.")
        probabilities += ENSEMBLE_WEIGHTS[name]*values
    if not np.isfinite(probabilities).all() or (probabilities < 0).any() or (probabilities > 1).any():
        raise ValueError("Invalid ensemble probabilities.")
    result = pd.DataFrame(probabilities, columns=template.columns.drop("Run No"))
    result.insert(0, "Run No", template["Run No"].to_numpy())
    directory = Path("outputs/hydro_ensemble") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ_submit")
    directory.mkdir(parents=True, exist_ok=False)
    path = directory / "ensemble_submission.csv"
    result.to_csv(path, index=False)
    checked = pd.read_csv(path)
    assert checked.shape == template.shape and checked.columns.equals(template.columns)
    np.testing.assert_array_equal(checked["Run No"], template["Run No"])
    np.testing.assert_allclose(checked.iloc[:, 1:].to_numpy(), probabilities, atol=1e-12, rtol=1e-12)
    elapsed = perf_counter()-ENSEMBLE_STARTED
    manifest = {"status": "complete", "weights": ENSEMBLE_WEIGHTS,
        "total_seconds": elapsed, "component_runs": COMPONENT_METADATA,
        "submission": str(path), "submission_sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    (directory/"run.json").write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding="utf-8")
    print("Verified ensemble:", path, "Total seconds:", elapsed)
